<h1 style="text-align:center;">Song Success Prediction Model</h1>
<h2 style="text-align:center;">Modeling</h2>

<h4 style="text-align:center;">by Cameron Hicks</h3>

V3 of the The Song Success Prediction Model project aims to create a Machine Learning Model that can predict which Country a song will be most popular in during the early or pre-release stages of production. The objective of this notebook is to train 3 Machine Learning Models and ultimately determine the best model to select as the final output of this project. In the steps below each model will be trained on the training data sets, then their accuracy will be tested using the test data sets. All three models will be objectively compared to make the final determination.

In [25]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    accuracy_score, 
    precision_score,
    recall_score,
    f1_score,
    auc,
    roc_curve,
    roc_auc_score,
    RocCurveDisplay,
    classification_report,
    ConfusionMatrixDisplay,
    mean_absolute_error, 
    mean_squared_error, 
    r2_score,
    precision_recall_curve
)

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Import Data

In [8]:
country_model_df = pd.read_csv('../../Data/V2_combined_df.csv')

In [9]:
country_model_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 84954 entries, 0 to 84953
Data columns (total 49 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   track_id            84954 non-null  object 
 1   track_name          84954 non-null  object 
 2   artist_name         84954 non-null  object 
 3   album_name          84954 non-null  object 
 4   release_date        84954 non-null  object 
 5   genre               84954 non-null  object 
 6   duration_ms         84954 non-null  int64  
 7   popularity          84954 non-null  int64  
 8   danceability        84954 non-null  float64
 9   energy              84954 non-null  float64
 10  key                 84954 non-null  int64  
 11  loudness            84954 non-null  float64
 12  mode                84954 non-null  object 
 13  instrumentalness    84954 non-null  float64
 14  tempo               84954 non-null  float64
 15  stream_count        84954 non-null  int64  
 16  coun

# Drop Unnecessary and Leakage Risk Data

In [12]:
cols_to_drop = ['track_id', 'track_name', 'artist_name', 'album_name', 'duration_mm:ss:ms', 'alpha2Code',
       'capital', 'subregion', 'region', 'population', 'latlng', 'demonym',
       'area', 'gini', 'independent', 'num_timezones', 'num_borders',
       'currency_code', 'num_languages', 'lang_english', 'lang_french',
       'lang_german', 'lang_japanese', 'lang_portuguese', 'lang_spanish',
       'bloc_AU', 'bloc_EU', 'bloc_NAFTA', 'bloc_PA', 'bloc_USAN', 'hit', 
        'days_since_release', 'streams_per_day']

country_model_df = country_model_df.drop(columns=cols_to_drop, axis=1)

In [18]:
country_model_df = country_model_df.drop('release_date', axis=1)

In [19]:
country_model_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 84954 entries, 0 to 84953
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   genre              84954 non-null  object 
 1   duration_ms        84954 non-null  int64  
 2   popularity         84954 non-null  int64  
 3   danceability       84954 non-null  float64
 4   energy             84954 non-null  float64
 5   key                84954 non-null  int64  
 6   loudness           84954 non-null  float64
 7   mode               84954 non-null  object 
 8   instrumentalness   84954 non-null  float64
 9   tempo              84954 non-null  float64
 10  stream_count       84954 non-null  int64  
 11  country_name       84954 non-null  object 
 12  explicit           84954 non-null  bool   
 13  label              84954 non-null  object 
 14  artist_song_count  84954 non-null  int64  
dtypes: bool(1), float64(5), int64(5), object(4)
memory usage: 9.2+ MB


# Train Test Split

In [20]:
# Separate data into X = all columns except target feature & y = target feature

X = country_model_df.drop(columns='country_name')
y = country_model_df['country_name']

In [21]:
cat_cols = X.select_dtypes(include='object').columns
num_cols = X.select_dtypes(include=['int64','float64']).columns
bool_cols = X.select_dtypes(include='bool').columns

In [22]:
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

In [23]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# Scale Features

In [24]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [27]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

In [7]:
def evaluate_classifier(model, X_test, y_test, model_name="Model"):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    auc = roc_auc_score(y_test, y_prob)

    print(f"\n=== {model_name} ===")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1 Score : {f1:.4f}")
    print(f"ROC AUC  : {auc:.4f}")

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, zero_division=0))

    ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
    plt.title(f"Confusion Matrix - {model_name}")
    plt.show()

    return {
        "Model": model_name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1": f1,
        "ROC_AUC": auc
    }

# Linear Regression

In [28]:
lr = LinearRegression()
lr.fit(X_train_scaled, y_train_encoded)

y_pred_lr = lr.predict(X_test_scaled)

In [30]:
print("LINEAR REGRESSION RESULTS")

r2 = r2_score(y_test_encoded, y_pred_lr)
mae = mean_absolute_error(y_test_encoded, y_pred_lr)
rmse = np.sqrt(mean_squared_error(y_test_encoded, y_pred_lr))

print("R2:", r2)
print("MAE:", mae)
print("RMSE:", rmse)

LINEAR REGRESSION RESULTS
R2: -6.969075127583046e-05
MAE: 2.4926924535603034
RMSE: 2.865369726828921


# Random Forest Regression

In [32]:
rf = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=1
)

rf.fit(X_train_scaled, y_train_encoded)

y_pred_rf = rf.predict(X_test_scaled)

In [33]:
print("\nRANDOM FOREST RESULTS")

r2 = r2_score(y_test_encoded, y_pred_rf)
mae = mean_absolute_error(y_test_encoded, y_pred_rf)
rmse = np.sqrt(mean_squared_error(y_test_encoded, y_pred_rf))

print("R2:", r2)
print("MAE:", mae)
print("RMSE:", rmse)


RANDOM FOREST RESULTS
R2: -0.01170309249705137
MAE: 2.4943911482549583
RMSE: 2.881987377112175


In [34]:
# Find the best parameters for Random Forest Regression

param_grid = {
    'n_estimators': [200, 400],
    'max_depth': [10, 20, None]
}

grid = GridSearchCV(
    RandomForestRegressor(random_state=42),
    param_grid,
    cv=3,
    scoring='r2',
    n_jobs=-1
)

grid.fit(X_train_scaled, y_train_encoded)

print(grid.best_params_)


{'max_depth': 10, 'n_estimators': 400}


In [35]:
rf2 = RandomForestRegressor(
    n_estimators=400,
    max_depth=10,
    random_state=42,
    n_jobs=1
)

rf2.fit(X_train_scaled, y_train_encoded)

y_pred_rf2 = rf2.predict(X_test_scaled)

In [37]:
print("RANDOM FOREST IMPROVED PARAMS RESULTS")

r2 = r2_score(y_test_encoded, y_pred_rf2)
mae = mean_absolute_error(y_test_encoded, y_pred_rf2)
rmse = np.sqrt(mean_squared_error(y_test_encoded, y_pred_rf2))

print("R2:", r2)
print("MAE:", mae)
print("RMSE:", rmse)

RANDOM FOREST IMPROVED PARAMS RESULTS
R2: -0.0005816195141379321
MAE: 2.4926102388187226
RMSE: 2.8661030144794992


# Gradient Boosting Regressor

In [38]:
gb = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

gb.fit(X_train_scaled, y_train_encoded)

y_pred_gb = gb.predict(X_test_scaled)

In [40]:
print("GRADIENT BOOSTING RESULTS")
print("-------------------------")

r2 = r2_score(y_test_encoded, y_pred_gb)
mae = mean_absolute_error(y_test_encoded, y_pred_gb)
rmse = np.sqrt(mean_squared_error(y_test_encoded, y_pred_gb))

print("R2:", r2)
print("MAE:", mae)
print("RMSE:", rmse)


GRADIENT BOOSTING RESULTS
-------------------------
R2: -0.0015636052799614664
MAE: 2.4921745157944004
RMSE: 2.8675090877603067


# V3 Determination

At this step in V3 of this project, we can see that each of the models received a negative R2 score to predict the Country a song will be popular in. We can determine at this step that the dataset does not have enough correlating features to train a model to accurately predict the target variable.